# 01 Data Collection & Cleaning
This notebook focuses on obtaining and preparing the official Central Bank of Sri Lanka (CBSL) USD/LKR exchange-rate series. 

The goal is to filter the raw dataset for the indicative exchange rate and ensure the data is cleaned and formatted for time-series analysis.

In [1]:
import pandas as pd

## 1. Loading the Dataset
First, we load the raw CBSL dataset and inspect its structure to understand the available columns and types of exchange rates.

In [2]:
cbsl = pd.read_csv("../data/raw/cbsl.csv")

In [3]:
cbsl.head()

,date,base,quote,type,value
0,2010-05-07,AED,LKR,indicative,30.9558
1,2010-05-07,AUD,LKR,indicative,101.1737
2,2010-05-07,BDT,LKR,indicative,1.6460
3,2010-05-07,BHD,LKR,indicative,301.6249
4,2010-05-07,BND,LKR,indicative,81.7791


In [4]:
cbsl.shape

(190935, 5)

In [5]:
cbsl.columns

Index(['date', 'base', 'quote', 'type', 'value'], dtype='str')

In [6]:
cbsl["type"].unique()

<StringArray>
['indicative']
Length: 1, dtype: str

In [7]:
cbsl.head()

,date,base,quote,type,value
0,2010-05-07,AED,LKR,indicative,30.9558
1,2010-05-07,AUD,LKR,indicative,101.1737
2,2010-05-07,BDT,LKR,indicative,1.6460
3,2010-05-07,BHD,LKR,indicative,301.6249
4,2010-05-07,BND,LKR,indicative,81.7791


## 2. Filtering for USD/LKR Indicative Rate
According to the project requirements, we only use the `indicative` rate for the USD to LKR pair.

In [8]:
usd_lkr = cbsl[
    (cbsl["base"] == "USD") &
    (cbsl["quote"] == "LKR") &
    (cbsl["type"] == "indicative")
].copy()

In [9]:
usd_lkr.shape

(3499, 5)

In [10]:
usd_lkr = (
    usd_lkr[["date", "value"]]
    .rename(columns={"value": "USD_LKR"})
)

In [11]:
usd_lkr.head()

,date,USD_LKR
47,2010-05-07,113.6975
98,2010-05-11,113.6292
149,2010-05-12,113.6854
200,2010-05-13,113.7009
251,2010-05-14,113.6573


## 3. Data Cleaning & Formatting
We clean the filtered data by:
- Selecting only the date and value.
- Renaming the value column to `USD_LKR`.
- Converting the date column to datetime objects.
- Sorting chronologically and removing duplicates.
- Setting the date as the index.

In [12]:
usd_lkr["date"] = pd.to_datetime(usd_lkr["date"])

In [13]:
usd_lkr = usd_lkr.sort_values("date")

In [14]:
usd_lkr = usd_lkr.drop_duplicates("date")

In [15]:
usd_lkr = usd_lkr.set_index("date")

In [16]:
usd_lkr.head()

,USD_LKR
date,
2010-05-07,113.6975
2010-05-11,113.6292
2010-05-12,113.6854
2010-05-13,113.7009
2010-05-14,113.6573


In [17]:
usd_lkr.isna().sum()

USD_LKR    0
dtype: int64

## 4. Data Verification
Before saving, we verify the dataset for missing values, check the date range, and ensure the general information is correct.

In [18]:
usd_lkr.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 3499 entries, 2010-05-07 to 2026-09-25
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   USD_LKR  3499 non-null   float64
dtypes: float64(1)
memory usage: 54.7 KB


In [19]:
usd_lkr.index.min()

Timestamp('2010-05-07 00:00:00')

## 5. Saving Processed Data
The final cleaned dataset is saved as `master_dataset.csv` for use in the EDA and modeling notebooks.

In [20]:
usd_lkr.to_csv("../data/processed/master_dataset.csv")